In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy.io import loadmat

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report

import joblib

%matplotlib inline

sns.set_style("whitegrid")

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
project_path = Path(
    r"C:\Users\suthi\Downloads\AI_Motor_Predictive_Maintenance"
)

datasets_path = project_path / "datasets"
results_path = project_path / "results"

csv_path = results_path / "complete_bearing_features.csv"

print("Project path:", project_path)
print("Dataset path:", datasets_path)
print("CSV path:", csv_path)

Project path: C:\Users\suthi\Downloads\AI_Motor_Predictive_Maintenance
Dataset path: C:\Users\suthi\Downloads\AI_Motor_Predictive_Maintenance\datasets
CSV path: C:\Users\suthi\Downloads\AI_Motor_Predictive_Maintenance\results\complete_bearing_features.csv


In [5]:
df = pd.read_csv(csv_path)

print("Dataset loaded successfully!")

print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (3183, 14)


,Mean,RMS,Standard_Deviation,Variance,Minimum,Maximum,Peak_to_Peak,Kurtosis,Skewness,Crest_Factor,Label,Window_ID,Source_File,Signal_ID
0,0.012005,0.073256,0.072266,0.005222,-0.272869,0.217794,0.490663,-0.045824,-0.137322,3.724845,Normal,0,Normal_0.mat,X097_DE_time
1,0.013116,0.074507,0.073344,0.005379,-0.272869,0.212370,0.485239,-0.116338,-0.010099,3.662321,Normal,1,Normal_0.mat,X097_DE_time
2,0.012790,0.079035,0.077993,0.006083,-0.226765,0.223010,0.449775,-0.184580,-0.062348,2.869171,Normal,2,Normal_0.mat,X097_DE_time
3,0.012148,0.077916,0.076964,0.005923,-0.226765,0.223010,0.449775,-0.150657,-0.103819,2.910361,Normal,3,Normal_0.mat,X097_DE_time
4,0.011959,0.074338,0.073370,0.005383,-0.202148,0.219255,0.421403,-0.221760,0.025960,2.949440,Normal,4,Normal_0.mat,X097_DE_time


In [7]:
print("Class Distribution:")
print(df["Label"].value_counts())

print("\nClass Percentages:")
print(
    df["Label"].value_counts(normalize=True) * 100
)

Class Distribution:
Label
Normal        2124
Outer_Race    1059
Name: count, dtype: int64

Class Percentages:
Label
Normal        66.7295
Outer_Race    33.2705
Name: proportion, dtype: float64


In [9]:
# Section 3 – Data quality verification
# Cell 5: Check missing values

print("Missing Values:")
print(df.isnull().sum())

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

Missing Values:
Mean                  0
RMS                   0
Standard_Deviation    0
Variance              0
Minimum               0
Maximum               0
Peak_to_Peak          0
Kurtosis              0
Skewness              0
Crest_Factor          0
Label                 0
Window_ID             0
Source_File           0
Signal_ID             0
dtype: int64

Total Missing Values:
0


In [11]:
# Cell 6: Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

print(
    "Duplicate signal windows:",
    df.duplicated(
        subset=["Source_File", "Signal_ID", "Window_ID"]
    ).sum()
)

Duplicate rows: 0
Duplicate signal windows: 0


Section 4 – Verify original recording metadata
Before splitting, we need to check the original .mat files.

This is important because:
- Normal_2.mat contains two Drive-End signals.
- Some recordings may have RPM information while others do not.
- Sampling frequency must be confirmed from the dataset's experimental documentation, not guessed from signal length.

In [14]:
mat_files = list(datasets_path.rglob("*.mat"))

print("Total MAT files found:", len(mat_files))

metadata = []

for file_path in mat_files:

    data = loadmat(file_path)

    de_keys = [
        key for key in data
        if key.endswith("_DE_time")
    ]

    rpm_keys = [
        key for key in data
        if "RPM" in key
    ]

    for key in de_keys:

        signal = data[key].squeeze()

        metadata.append({
            "Source_File": file_path.name,
            "Signal_ID": key,
            "Samples": len(signal),
            "RPM_Variable": ", ".join(rpm_keys) if rpm_keys else "Unavailable"
        })

metadata_df = pd.DataFrame(metadata)

display(metadata_df)

print("\nTotal Drive-End signals:", len(metadata_df))

Total MAT files found: 17


,Source_File,Signal_ID,Samples,RPM_Variable
0,B007_0.mat,X118_DE_time,122571,X118RPM
1,B007_1HP.mat,X119_DE_time,121410,X119RPM
2,IR007_0HP.mat,X105_DE_time,121265,X105RPM
3,IR007_1HP.mat,X106_DE_time,121991,X106RPM
4,Normal_0.mat,X097_DE_time,243938,X097RPM
5,Normal_1.mat,X098_DE_time,483903,Unavailable
6,Normal_2.mat,X098_DE_time,483903,Unavailable
7,Normal_2.mat,X099_DE_time,485063,Unavailable
8,Normal_3.mat,X100_DE_time,485643,X100RPM
9,0R007_12_0HP.mat,X156_DE_time,122281,X156RPM



Total Drive-End signals: 18


In [16]:
recording_summary = (
    df.groupby(["Source_File", "Label"])
    .size()
    .reset_index(name="Number_of_Windows")
)

display(recording_summary)

print(
    "\nNumber of unique original recordings:",
    df["Source_File"].nunique()
)

,Source_File,Label,Number_of_Windows
0,0R007_12_0HP.mat,Outer_Race,118
1,0R007_12_1HP.mat,Outer_Race,118
2,0R007_3_0HP.mat,Outer_Race,118
3,0R007_3_1HP.mat,Outer_Race,117
4,0R007_6_0HP.mat,Outer_Race,118
5,0R007_6_1HP.mat,Outer_Race,118
6,0R021_12_0HP.mat,Outer_Race,117
7,0R021_6_0HP.mat,Outer_Race,118
8,Normal_0.mat,Normal,237
9,Normal_1.mat,Normal,471



Number of unique original recordings: 13


Section 5 – Separate features and target
Now we separate the information the AI model will learn from and what it needs to predict.
- X: Vibration features (inputs).
- y: Bearing condition (output).
We must exclude Window_ID, Source_File and Signal_ID from model inputs. They are identifiers, not vibration features.

In [19]:
feature_columns = [
    "Mean",
    "RMS",
    "Standard_Deviation",
    "Variance",
    "Minimum",
    "Maximum",
    "Peak_to_Peak",
    "Kurtosis",
    "Skewness",
    "Crest_Factor"
]

X = df[feature_columns].copy()

y = df["Label"].copy()

groups = df["Source_File"].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Groups shape:", groups.shape)

print("\nInput features:")
print(X.columns.tolist())

Feature shape: (3183, 10)
Target shape: (3183,)
Groups shape: (3183,)

Input features:
['Mean', 'RMS', 'Standard_Deviation', 'Variance', 'Minimum', 'Maximum', 'Peak_to_Peak', 'Kurtosis', 'Skewness', 'Crest_Factor']


In [21]:
# Cell 10: Encode target labels
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Class mapping:")

for class_name, encoded_value in zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
):
    print(class_name, "=", encoded_value)

Class mapping:
Normal = 0
Outer_Race = 1


Section 6 – Split training and testing data
This is the most important section of Notebook 04.

Normally, developers might use a random 80:20 split. However, that would be dangerous for our project because multiple windows come from the same original recording.

Instead, we will use StratifiedGroupKFold.

It helps preserve class representation while keeping each original recording entirely within one split.

In [26]:
# Cell 11: Create group-based train-test splits
splitter = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

splits = list(
    splitter.split(X, y_encoded, groups)
)

print("Number of splits:", len(splits))

for fold, (train_idx, test_idx) in enumerate(splits):

    print(f"\nFold {fold + 1}")

    print("Training windows:", len(train_idx))
    print("Testing windows:", len(test_idx))

    print(
        "Training recordings:",
        groups.iloc[train_idx].nunique()
    )

    print(
        "Testing recordings:",
        groups.iloc[test_idx].nunique()
    )

    print(
        "Test class distribution:",
        y.iloc[test_idx].value_counts().to_dict()
    )

Number of splits: 4

Fold 1
Training windows: 1650
Testing windows: 1533
Training recordings: 10
Testing recordings: 3
Test class distribution: {'Normal': 1416, 'Outer_Race': 117}

Fold 2
Training windows: 2712
Testing windows: 471
Training recordings: 9
Testing recordings: 4
Test class distribution: {'Outer_Race': 471}

Fold 3
Training windows: 2476
Testing windows: 707
Training recordings: 10
Testing recordings: 3
Test class distribution: {'Normal': 471, 'Outer_Race': 236}

Fold 4
Training windows: 2711
Testing windows: 472
Training recordings: 10
Testing recordings: 3
Test class distribution: {'Normal': 237, 'Outer_Race': 235}


In [28]:
# first fold
train_idx, test_idx = splits[0]

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y_encoded[train_idx]
y_test = y_encoded[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (1650, 10)
Testing shape: (1533, 10)


In [64]:
print("Original training shape:", X_train.shape)
print("Original testing shape:", X_test.shape)

print("\nOriginal training recordings:")
print(groups_train.unique())

print("\nOriginal testing recordings:")
print(groups_test.unique())

print("Training class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts())

Original training shape: (2710, 10)
Original testing shape: (473, 10)

Original training recordings:
['Normal_1.mat' 'Normal_2.mat' 'Normal_3.mat' '0R007_3_0HP.mat'
 '0R007_3_1HP.mat' '0R007_6_0HP.mat' '0R007_6_1HP.mat' '0R021_12_0HP.mat'
 '0R021_6_0HP.mat' 'OR021_3_0HP.mat']

Original testing recordings:
['Normal_0.mat' '0R007_12_0HP.mat' '0R007_12_1HP.mat']
Training class distribution:
0    1887
1     823
Name: count, dtype: int64

Testing class distribution:
0    237
1    236
Name: count, dtype: int64


In [30]:
# verify no leakage
train_recordings = set(groups_train)
test_recordings = set(groups_test)

common_recordings = train_recordings.intersection(
    test_recordings
)

print("Training recordings:", train_recordings)
print("Testing recordings:", test_recordings)

print("Common recordings:", common_recordings)

assert len(common_recordings) == 0

print("\nNo recording leakage detected!")

Training recordings: {'0R021_6_0HP.mat', '0R007_3_0HP.mat', '0R007_6_1HP.mat', 'OR021_3_0HP.mat', '0R007_6_0HP.mat', 'Normal_0.mat', '0R007_12_1HP.mat', '0R007_12_0HP.mat', '0R007_3_1HP.mat', 'Normal_1.mat'}
Testing recordings: {'Normal_2.mat', 'Normal_3.mat', '0R021_12_0HP.mat'}
Common recordings: set()

No recording leakage detected!


In [32]:
# verify class dist after splitting
train_labels = label_encoder.inverse_transform(y_train)
test_labels = label_encoder.inverse_transform(y_test)

print("Training class distribution:")
print(pd.Series(train_labels).value_counts())

print("\nTesting class distribution:")
print(pd.Series(test_labels).value_counts())

Training class distribution:
Outer_Race    942
Normal        708
Name: count, dtype: int64

Testing class distribution:
Normal        1416
Outer_Race     117
Name: count, dtype: int64


Section 7 – Feature scaling
Our features have different numerical ranges.

For example:
- Mean is relatively small.
- RMS can be much larger.
- Kurtosis can reach values above 40.
Some machine-learning algorithms are sensitive to these differences.

We'll use StandardScaler to standardize the features.

The formula is:
\[
z = \frac{x-\mu}{\sigma}
\]

Where:
- \(x\) = original feature value
- \(\mu\) = training-set mean
- \(\sigma\) = training-set standard deviation

In [35]:
# apply standardscaler
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform testing data using training statistics
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")

print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

Feature scaling completed!
Training shape: (1650, 10)
Testing shape: (1533, 10)


In [37]:
scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns
)

print("Training feature means:")
display(scaled_df.mean().round(3))

print("\nTraining feature standard deviations:")
display(scaled_df.std().round(3))

Training feature means:


Mean                  0.0
RMS                   0.0
Standard_Deviation    0.0
Variance             -0.0
Minimum               0.0
Maximum              -0.0
Peak_to_Peak          0.0
Kurtosis              0.0
Skewness              0.0
Crest_Factor          0.0
dtype: float64


Training feature standard deviations:


Mean                  1.0
RMS                   1.0
Standard_Deviation    1.0
Variance              1.0
Minimum               1.0
Maximum               1.0
Peak_to_Peak          1.0
Kurtosis              1.0
Skewness              1.0
Crest_Factor          1.0
dtype: float64

Section 8 – Save processed datasets
We'll save the processed datasets so Notebook 05 can load them directly.

In [46]:
processed_path = results_path / "processed"

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

print("Processed folder created:", processed_path)

Processed folder created: C:\Users\suthi\Downloads\AI_Motor_Predictive_Maintenance\results\processed


In [42]:
train_df = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns
)

train_df["Label"] = y_train
train_df["Source_File"] = groups_train.values

test_df = pd.DataFrame(
    X_test_scaled,
    columns=feature_columns
)

test_df["Label"] = y_test
test_df["Source_File"] = groups_test.values

train_df.to_csv(
    processed_path / "train_data.csv",
    index=False
)

test_df.to_csv(
    processed_path / "test_data.csv",
    index=False
)

print("Training and testing datasets saved!")

Training and testing datasets saved!


In [44]:
joblib.dump(
    scaler,
    processed_path / "standard_scaler.pkl"
)

joblib.dump(
    label_encoder,
    processed_path / "label_encoder.pkl"
)

print("Scaler and label encoder saved!")

Scaler and label encoder saved!


In [48]:
print("========== FINAL DATASET VERIFICATION ==========")

print("\nTraining dataset:")
print("Shape:", train_df.shape)
print("Classes:")
print(train_df["Label"].value_counts())

print("\nTesting dataset:")
print("Shape:", test_df.shape)
print("Classes:")
print(test_df["Label"].value_counts())

print("\nNumber of training recordings:",
      train_df["Source_File"].nunique())

print("Number of testing recordings:",
      test_df["Source_File"].nunique())

print("\nMissing values:")
print("Training:", train_df.isnull().sum().sum())
print("Testing:", test_df.isnull().sum().sum())

assert set(train_df["Label"].unique()) == {0, 1}
assert set(test_df["Label"].unique()) == {0, 1}
assert not train_df.isnull().any().any()
assert not test_df.isnull().any().any()

print("\nAll verification checks passed!")

========== FINAL DATASET VERIFICATION ==========

Training dataset:
Shape: (1650, 12)
Classes:
Label
1    942
0    708
Name: count, dtype: int64

Testing dataset:
Shape: (1533, 12)
Classes:
Label
0    1416
1     117
Name: count, dtype: int64

Number of training recordings: 10
Number of testing recordings: 3

Missing values:
Training: 0
Testing: 0

All verification checks passed!


In [50]:
# diagnostic cell
print("TRAINING RECORDING DISTRIBUTION")

print(
    train_df.groupby(
        ["Source_File", "Label"]
    ).size()
)

print("\nTESTING RECORDING DISTRIBUTION")

print(
    test_df.groupby(
        ["Source_File", "Label"]
    ).size()
)

TRAINING RECORDING DISTRIBUTION
Source_File       Label
0R007_12_0HP.mat  1        118
0R007_12_1HP.mat  1        118
0R007_3_0HP.mat   1        118
0R007_3_1HP.mat   1        117
0R007_6_0HP.mat   1        118
0R007_6_1HP.mat   1        118
0R021_6_0HP.mat   1        118
Normal_0.mat      0        237
Normal_1.mat      0        471
OR021_3_0HP.mat   1        117
dtype: int64

TESTING RECORDING DISTRIBUTION
Source_File       Label
0R021_12_0HP.mat  1        117
Normal_2.mat      0        943
Normal_3.mat      0        473
dtype: int64


In [52]:
# Select original recordings for testing

test_recordings = [
    "Normal_0.mat",
    "0R007_12_0HP.mat",
    "0R007_12_1HP.mat"
]

# Create masks

test_mask = df["Source_File"].isin(test_recordings)

train_mask = ~test_mask

# Separate raw features and labels

X_train = df.loc[train_mask, feature_columns].copy()
X_test = df.loc[test_mask, feature_columns].copy()

y_train = label_encoder.transform(
    df.loc[train_mask, "Label"]
)

y_test = label_encoder.transform(
    df.loc[test_mask, "Label"]
)

groups_train = df.loc[train_mask, "Source_File"]
groups_test = df.loc[test_mask, "Source_File"]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (2710, 10)
Testing shape: (473, 10)


In [54]:
print("TRAINING DISTRIBUTION")

print(
    df.loc[train_mask, "Label"].value_counts()
)

print("\nTESTING DISTRIBUTION")

print(
    df.loc[test_mask, "Label"].value_counts()
)

TRAINING DISTRIBUTION
Label
Normal        1887
Outer_Race     823
Name: count, dtype: int64

TESTING DISTRIBUTION
Label
Normal        237
Outer_Race    236
Name: count, dtype: int64


In [56]:
common_recordings = (
    set(groups_train).intersection(set(groups_test))
)

print("Common recordings:", common_recordings)

assert len(common_recordings) == 0

print("No recording leakage detected!")

Common recordings: set()
No recording leakage detected!


In [58]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Training scaled:", X_train_scaled.shape)
print("Testing scaled:", X_test_scaled.shape)

Training scaled: (2710, 10)
Testing scaled: (473, 10)


In [60]:
train_df = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns
)

train_df["Label"] = y_train
train_df["Source_File"] = groups_train.values

test_df = pd.DataFrame(
    X_test_scaled,
    columns=feature_columns
)

test_df["Label"] = y_test
test_df["Source_File"] = groups_test.values

train_df.to_csv(
    processed_path / "train_data.csv",
    index=False
)

test_df.to_csv(
    processed_path / "test_data.csv",
    index=False
)

joblib.dump(
    scaler,
    processed_path / "standard_scaler.pkl"
)

joblib.dump(
    label_encoder,
    processed_path / "label_encoder.pkl"
)

print("Corrected datasets saved successfully!")

Corrected datasets saved successfully!
